In [1]:
# Group suffixes into natural categories

import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import parse_casenum

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [2]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances.parquet"))
sfx_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc", "suffix_groups.csv"))

In [3]:
SFX_MAP = {}
for _, row in sfx_df.iterrows():
    sfx = row['suffix']
    grp = row['category_code']
    SFX_MAP[sfx] = grp

In [4]:
for sfx in sfx_df['suffix'].unique():
    df[f'sfx_{sfx}'] = False
for grp in sfx_df['category_code'].unique():
    df[f'sfx_grp_{grp}'] = False

for idx, row in df.iterrows():
    casenum = row['title']
    parsed_casenum = parse_casenum(casenum)
    suffixes = parsed_casenum['suffixes']
    for sfx in suffixes:
        df.loc[idx, f'sfx_{sfx}'] = True
        if sfx in SFX_MAP:
            df.loc[idx, f'sfx_grp_{SFX_MAP[sfx]}'] = True

In [5]:
df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances_suffixes.parquet"))